# Подсчёт рыночных заявок и связанных edge cases в OrderLog

Считаем по всем дням в `data/raw/`:
- ACTION=1, PRICE=0 — рыночные заявки (приход)
- ACTION=2, PRICE=0 — сделки против рыночных (для контроля)
- доли от ACTION=1 и от общего числа строк
- разбивка по инструментам (топ-10) — чтобы понять, концентрируются ли market orders в каких-то бумагах

Цель: оценить, какую долю данных мы потеряем, если просто выкинем рыночные заявки при конверсии в hftbacktest L3.

In [1]:
from pathlib import Path
import polars as pl

RAW_DIR = Path("../data/raw")
files = sorted(RAW_DIR.glob("OrderLog2024*.txt"))
print(f"Found {len(files)} files:")
for f in files:
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

Found 5 files:
  OrderLog20240318.txt (5118.9 MB)
  OrderLog20240319.txt (6980.7 MB)
  OrderLog20240320.txt (7787.1 MB)
  OrderLog20240321.txt (5690.5 MB)
  OrderLog20240322.txt (6057.7 MB)


In [2]:
# Per-day aggregate counts via lazy scan.
# Concatenate, then group by source file path so we get one row per day.

DTYPES = {
    "NO": pl.Int64,
    "SECCODE": pl.Utf8,
    "BUYSELL": pl.Utf8,
    "TIME": pl.Int64,
    "ORDERNO": pl.Int64,
    "ACTION": pl.Int8,
    "PRICE": pl.Float64,
    "VOLUME": pl.Int64,
    "TRADENO": pl.Float64,
    "TRADEPRICE": pl.Float64,
}

rows = []
for f in files:
    lf = pl.scan_csv(str(f), schema_overrides=DTYPES)
    stats = (
        lf.select(
            total=pl.len(),
            action0=(pl.col("ACTION") == 0).sum(),
            action1=(pl.col("ACTION") == 1).sum(),
            action2=(pl.col("ACTION") == 2).sum(),
            market_new=((pl.col("ACTION") == 1) & (pl.col("PRICE") == 0)).sum(),
            trade_p0=((pl.col("ACTION") == 2) & (pl.col("PRICE") == 0)).sum(),
        )
        .collect()
        .row(0, named=True)
    )
    stats["file"] = f.name
    stats["market_pct_of_action1"] = 100.0 * stats["market_new"] / max(stats["action1"], 1)
    stats["market_pct_of_total"] = 100.0 * stats["market_new"] / max(stats["total"], 1)
    rows.append(stats)

summary = pl.DataFrame(rows).select(
    "file", "total", "action0", "action1", "action2",
    "market_new", "market_pct_of_action1", "market_pct_of_total", "trade_p0"
)
summary

file,total,action0,action1,action2,market_new,market_pct_of_action1,market_pct_of_total,trade_p0
str,i64,i64,i64,i64,i64,f64,f64,i64
"""OrderLog20240318.txt""",94391716,40653590,45217306,8520820,462497,1.022832,0.489976,877590
"""OrderLog20240319.txt""",128884203,57599676,62359040,8925487,532396,0.853759,0.413081,1063682
"""OrderLog20240320.txt""",143745627,65394554,69922825,8428248,518522,0.741563,0.360722,1050849
"""OrderLog20240321.txt""",105503218,47026525,51054429,7422264,420931,0.824475,0.398975,845722
"""OrderLog20240322.txt""",112018713,50474959,54360057,7183697,386921,0.711774,0.345407,777216


In [3]:
# Aggregate across all days
totals = summary.select(
    pl.col("total").sum().alias("total_rows"),
    pl.col("action1").sum().alias("total_action1"),
    pl.col("market_new").sum().alias("total_market_new"),
    pl.col("trade_p0").sum().alias("total_trade_p0"),
)
tot = totals.row(0, named=True)
print(f"Across all {len(files)} days:")
print(f"  rows total:                      {tot['total_rows']:>15,}")
print(f"  ACTION=1 (new orders):           {tot['total_action1']:>15,}")
print(f"  ACTION=1 & PRICE=0 (market new): {tot['total_market_new']:>15,}  ({100*tot['total_market_new']/tot['total_action1']:.4f}% of action1, {100*tot['total_market_new']/tot['total_rows']:.4f}% of all rows)")
print(f"  ACTION=2 & PRICE=0 (trade vs market resting): {tot['total_trade_p0']:>10,}")

Across all 5 days:
  rows total:                          584,543,477
  ACTION=1 (new orders):               282,913,657
  ACTION=1 & PRICE=0 (market new):       2,321,267  (0.8205% of action1, 0.3971% of all rows)
  ACTION=2 & PRICE=0 (trade vs market resting):  4,615,059


In [4]:
# Top instruments by market-order count (across all days combined)
per_inst = []
for f in files:
    lf = pl.scan_csv(str(f), schema_overrides=DTYPES)
    df = (
        lf.filter((pl.col("ACTION") == 1) & (pl.col("PRICE") == 0))
        .group_by("SECCODE")
        .len()
        .rename({"len": "market_new"})
        .collect()
    )
    per_inst.append(df)

by_inst = (
    pl.concat(per_inst)
    .group_by("SECCODE")
    .agg(pl.col("market_new").sum())
    .sort("market_new", descending=True)
)

# Add share of total ACTION=1 per instrument
per_inst_action1 = []
for f in files:
    lf = pl.scan_csv(str(f), schema_overrides=DTYPES)
    df = (
        lf.filter(pl.col("ACTION") == 1)
        .group_by("SECCODE")
        .len()
        .rename({"len": "action1"})
        .collect()
    )
    per_inst_action1.append(df)

by_inst_action1 = (
    pl.concat(per_inst_action1)
    .group_by("SECCODE")
    .agg(pl.col("action1").sum())
)

joined = (
    by_inst.join(by_inst_action1, on="SECCODE", how="left")
    .with_columns(pct_of_action1=100.0 * pl.col("market_new") / pl.col("action1"))
    .sort("market_new", descending=True)
)

print("Top-15 instruments by market-order count:")
joined.head(15)

Top-15 instruments by market-order count:


SECCODE,market_new,action1,pct_of_action1
str,u32,u32,f64
"""TCSG""",224981,9570934,2.350669
"""ROSN""",108029,2466999,4.378964
"""YNDX""",76714,44240003,0.173404
"""SBER""",72084,5959710,1.209522
"""SFIN""",62694,24221377,0.258837
…,…,…,…
"""VKCO""",46092,1383762,3.33092
"""LQDT""",45426,552625,8.220041
"""RNFT""",42120,757824,5.558019


In [7]:
!export PYTHONPATH=/Users/mem4/Documents/tradefm

In [15]:
# Among the 20 instruments we actually train on, how many market orders?
# (This is the practically relevant number — we only convert these to hftbacktest.)
from src.config import PipelineConfig
from src.data.loader import discover_files, select_instruments

cfg = PipelineConfig()
cfg.raw_dir = str(RAW_DIR.resolve())
selected_files = discover_files(cfg)
selected_instruments = select_instruments(selected_files, cfg)
print(f"Selected {len(selected_instruments)} instruments: {selected_instruments[:5]} ...")

in_selected = joined.filter(pl.col("SECCODE").is_in(selected_instruments))
print("\nMarket orders among selected (top-20) instruments:")
in_selected

ModuleNotFoundError: No module named 'src'

## Интерпретация

- `market_pct_of_action1` < 1% → можно спокойно выкидывать, потерь почти нет
- 1–5% → выкидывать, но заметить в коммите конвертера
- > 5% → надо обрабатывать честно (синтезировать FILL'ы против top of book)

Также сравнить `market_new` с `trade_p0`: каждой рыночной заявке должно соответствовать ≥1 трейд против неё (с её стороны OrderLog видит её PRICE=0). Так что `trade_p0` ≈ Σ(сделок-исполнений рыночных заявок). Если они близки по порядку — картина согласованная.